In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [2]:
# Pilot study data

data = {
    "name": [],
    "gender": [],
    "ethnicity": [],
    "score": []
}

# Vi bruger gennemsnittet af lav, mellem og høj
# fordi CV-niveau ikke er det, vi undersøger

pilot_data = {
    "Aisha": ["Female", "Arabic", [72, 68, 65], [78, 82, 76], [84, 84, 86]],
    "Maria": ["Female", "Danish", [68, 64, 62], [82, 78, 82], [84, 86, 86]],
    "Peter": ["Male", "Danish", [66, 58, 61], [74, 74, 78], [86, 86, 86]],
    "Mohammed": ["Male", "Arabic", [68, 66, 64], [82, 82, 78], [86, 88, 86]]
}

for name in pilot_data:
    gender = pilot_data[name][0]
    ethnicity = pilot_data[name][1]
    low = pilot_data[name][2]
    medium = pilot_data[name][3]
    high = pilot_data[name][4]
    
    for i in range(3):
        average_score = (low[i] + medium[i] + high[i]) / 3
        
        data["name"].append(name)
        data["gender"].append(gender)
        data["ethnicity"].append(ethnicity)
        data["score"].append(average_score)

df = pd.DataFrame(data)

df

,name,gender,ethnicity,score
0,Aisha,Female,Arabic,78.000000
1,Aisha,Female,Arabic,78.000000
2,Aisha,Female,Arabic,75.666667
3,Maria,Female,Danish,78.000000
4,Maria,Female,Danish,76.000000
5,Maria,Female,Danish,76.666667
6,Peter,Male,Danish,75.333333
7,Peter,Male,Danish,72.666667
8,Peter,Male,Danish,75.000000
9,Mohammed,Male,Arabic,78.666667


In [3]:
model = ols("score ~ C(gender) * C(ethnicity)", data=df).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

anova_table

,sum_sq,df,F,PR(>F)
C(gender),3.000000,1.0,1.636364,0.236679
C(ethnicity),10.703704,1.0,5.838384,0.042094
C(gender):C(ethnicity),7.259259,1.0,3.959596,0.081789
Residual,14.666667,8.0,NaN,NaN


In [4]:
# Residual betyder "fejlvariationen"
# Det er variationen, som modellen ikke kan forklare

ss_error = anova_table.loc["Residual", "sum_sq"]

effect_sizes = {}

for effect in ["C(gender)", "C(ethnicity)", "C(gender):C(ethnicity)"]:
    
    ss_effect = anova_table.loc[effect, "sum_sq"]
    
    partial_eta_squared = ss_effect / (ss_effect + ss_error)
    
    cohens_f = np.sqrt(partial_eta_squared / (1 - partial_eta_squared))
    
    effect_sizes[effect] = cohens_f
    
    print(effect)
    print("Partial eta squared:", round(partial_eta_squared, 3))
    print("Cohen's f:", round(cohens_f, 3))
    print()

C(gender)
Partial eta squared: 0.17
Cohen's f: 0.452

C(ethnicity)
Partial eta squared: 0.422
Cohen's f: 0.854

C(gender):C(ethnicity)
Partial eta squared: 0.331
Cohen's f: 0.704



In [5]:
effect_size = min(effect_sizes.values())

print("Smallest effect size:", round(effect_size, 3))

Smallest effect size: 0.452
